## Cell 1 — Setup & imports

We load spaCy's small English model. It ships with tokenizer, POS tagger,
lemmatizer and a Named Entity Recognizer already trained.

# Lab 1 — spaCy NLP Pipeline

**Learning objective:** Build a complete text processing pipeline — tokenize, lemmatize, remove stop words, extract named entities — and visualise NER output.

**Concept link:** Slide 4 (NLP pipeline) and Slide 6 (NLP tasks)

**Time:** ~10 minutes

In [1]:
import spacy
from spacy import displacy
from pathlib import Path

# Load the small English pipeline. This fails with a clear OSError if the
# model hasn't been downloaded yet (see README for the download command).
nlp = spacy.load("en_core_web_sm")

print(f"spaCy version: {spacy.__version__}")
print(f"Model: {nlp.meta['name']} v{nlp.meta['version']}")

spaCy version: 3.7.4
Model: core_web_sm v3.7.1


## Cell 2 — Load the sample text

A short business-news paragraph mentioning organisations, a person, a
location and a date — good variety for demonstrating Named Entity Recognition.

In [2]:
text = Path("data/sample_text.txt").read_text()

print(f"Loaded {len(text)} characters")
print(text[:200])

Loaded 435 characters
Tata Consultancy Services (TCS), headquartered in Mumbai, reported a net profit of
â‚¹12,434 crore for Q1 FY2025, a 9% increase year-on-year. CEO K Krithivasan
addressed analysts on 11 July 2024, citi


In [3]:
## Cell 3 — Run the pipeline

#Calling `nlp(text)` runs the *entire* pipeline in one shot: tokenization,
#POS tagging, lemmatization, dependency parsing and NER all happen here.

In [4]:
doc = nlp(text)

print(f"Tokens: {len(doc)}")
print(f"Sentences: {len(list(doc.sents))}")

Tokens: 92
Sentences: 4


## Cell 4 — Token table

For every alphabetic token, print its text, lemma (dictionary/base form),
part-of-speech tag, and whether it's a stop word. This is the classic
"tokenize -> lemmatize -> remove stop words" pipeline made visible.

In [5]:
print(f"{'TOKEN':<20} {'LEMMA':<20} {'POS':<10} {'STOP':<6} {'ALPHA':<6}")
print("-" * 65)

# is_alpha=True filters out numbers, punctuation and currency symbols so
# the table only shows "real words".
for token in doc:
    if token.is_alpha:
        print(f"{token.text:<20} {token.lemma_:<20} {token.pos_:<10} "
              f"{str(token.is_stop):<6} {str(token.is_alpha):<6}")

TOKEN                LEMMA                POS        STOP   ALPHA 
-----------------------------------------------------------------
Tata                 Tata                 PROPN      False  True  
Consultancy          Consultancy          PROPN      False  True  
Services             Services             PROPN      False  True  
TCS                  TCS                  PROPN      False  True  
headquartered        headquarter          VERB       False  True  
in                   in                   ADP        True   True  
Mumbai               Mumbai               PROPN      False  True  
reported             report               VERB       False  True  
a                    a                    DET        True   True  
net                  net                  ADJ        False  True  
profit               profit               NOUN       False  True  
of                   of                   ADP        True   True  
crore                crore                NOUN       False  Tru

## Cell 5 — Named entity extraction

`doc.ents` gives every named entity spaCy detected, along with a label
(ORG, PERSON, GPE = geo-political entity, DATE, NORP = nationality/religious/
political group, etc.).

In [6]:
print(f"{'ENTITY':<30} {'LABEL':<12} {'START':>6} {'END':>6}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {ent.start_char:>6} {ent.end_char:>6}")

# Summarise how many entities of each type were found.
from collections import Counter
label_counts = Counter(ent.label_ for ent in doc.ents)
print(f"\nEntity type summary: {dict(label_counts)}")

ENTITY                         LABEL         START    END
------------------------------------------------------------
Tata Consultancy Services      ORG               0     25
TCS                            ORG              27     30
Mumbai                         GPE              50     56
Q1 FY2025                      PRODUCT         103    112
9%                             PERCENT         116    118
year-on-year                   DATE            128    140
K Krithivasan                  PERSON          146    159
11 July 2024                   DATE            182    194
North American                 NORP            222    236
40,000                         CARDINAL        272    278
FY2025                         FAC             291    297
Infosys                        ORG             349    356
Bengaluru                      GPE             367    376
2.1%                           PERCENT         410    414

Entity type summary: {'ORG': 3, 'GPE': 2, 'PRODUCT': 1, 'PERCENT': 2

**Expected entities:** TCS (ORG), Mumbai (GPE), K Krithivasan (PERSON),
11 July 2024 (DATE), North American (NORP), Infosys (ORG), Bengaluru (GPE).

## Cell 6 — Sentence similarity

spaCy averages the word vectors of a span to get a sentence vector, then
computes cosine similarity between two such vectors.

**Note:** `en_core_web_sm` does not ship full-size word vectors, so this
similarity score is only a rough approximation. `en_core_web_lg` (or a
dedicated sentence-embedding model, see Lab 2) gives far better results.

In [7]:
sentences = list(doc.sents)

if len(sentences) >= 2:
    s1, s2 = sentences[0], sentences[1]
    similarity = s1.similarity(s2)
    print(f"Sentence 1: {s1.text[:60]}...")
    print(f"Sentence 2: {s2.text[:60]}...")
    print(f"Cosine similarity: {similarity:.4f}")
    print("(Note: en_core_web_sm has small vectors — use en_core_web_lg for better similarity)")

Sentence 1: Tata Consultancy Services (TCS), headquartered in Mumbai, re...
Sentence 2: CEO K Krithivasan
addressed analysts on 11 July 2024, citing...
Cosine similarity: 0.5916
(Note: en_core_web_sm has small vectors — use en_core_web_lg for better similarity)


C:\Users\Admin\AppData\Local\Temp\ipykernel_24032\963320901.py:5: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Span.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarity = s1.similarity(s2)


## Cell 7 — Visualise NER with displacy

`displacy` renders the document as HTML with each entity highlighted and
colour-coded by type.

In [8]:
# Render inline in the notebook.
import sys
from IPython import display
sys.modules['IPython.core.display'] = display
displacy.render(doc, style="ent", jupyter=True)

## Cell 8 — Challenge (optional, ~2 minutes)

Strip out stop words and punctuation, then find the 10 most frequent
lemmas. Financial/business terms should dominate the list.

In [9]:
from collections import Counter

# Keep only meaningful words: no stop words, no punctuation, alphabetic only.
lemmas = [token.lemma_.lower() for token in doc
          if not token.is_stop and not token.is_punct and token.is_alpha]

print(Counter(lemmas).most_common(10))

[('report', 2), ('year', 2), ('tata', 1), ('consultancy', 1), ('services', 1), ('tcs', 1), ('headquarter', 1), ('mumbai', 1), ('net', 1), ('profit', 1)]
